# Step 0: vLLM feasibility on Kaggle T4 (vllm-feasibility)

**Settings:** Accelerator = **GPU T4 x2**, Internet = **ON** (benchmark notebook, not a submission)

**Input:** Your Work / shared -> Notebooks -> **llm-cls-train-v2** (our classifier)

**Question:** can vLLM run what Step 2 needs on Kaggle's T4 GPUs (older Turing GPUs, compute capability 7.5)?

| Test | Checks |
|---|---|
| env | vLLM installs, sees both GPUs |
| small_gen | Basic text generation (Qwen2.5-0.5B) |
| judge7b | 7B AWQ generative judge on 1 GPU |
| judge7b_tp2 | Same 7B split across 2 GPUs (tensor parallel), optional |
| classify | Our fine-tuned classifier served by vLLM |
| hf_match | vLLM classifier probabilities match HF (correctness) |

Every test runs as a **separate Python process**, so one crash doesn't kill the others and GPU memory is freed between tests. **Run All** (~45 min). The last cell prints a PASS / FAIL table.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!pip install -q -U vllm 2>&1 | tail -n 3
# vLLM installs its own PyTorch; Kaggle's old torchaudio then mismatches -> remove it (we don't need audio)
!pip uninstall -y -q torchaudio
!pip list 2>/dev/null | grep -E "^(vllm|torch|torchvision|torchaudio|transformers) "
!mkdir -p /kaggle/working/feas/logs

In [ ]:
%%writefile /kaggle/working/feas/common.py
import json, os, glob, time, traceback

OUT = "/kaggle/working/feas"

def record(name, status, **metrics):
    json.dump(dict(test=name, status=status, **metrics),
              open(f"{OUT}/{name}.json", "w"), indent=1, default=str)
    print(f"RESULT {name}: {status} {metrics}")

def run(name, fn):
    t = time.time()
    try:
        metrics = fn() or {}
        record(name, "PASS", seconds=round(time.time() - t, 1), **metrics)
    except Exception as e:
        traceback.print_exc()
        record(name, "FAIL", seconds=round(time.time() - t, 1), error=repr(e)[:400])

def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"input not found: {pattern} (Add Input llm-cls-train-v2?)"
    return hits[0]

def build_ids(tok, d, max_p, max_r):
    # Same input format as training (v1-default)
    H = lambda s: tok(s, add_special_tokens=False)["input_ids"]
    H_P, H_A, H_B, H_Q = H("Prompt: "), H("\n\nResponse A: "), H("\n\nResponse B: "), H("\n\nWhich response is better?")
    P = tok(d["prompt"].tolist(),     add_special_tokens=False, truncation=True, max_length=max_p)["input_ids"]
    A = tok(d["response_a"].tolist(), add_special_tokens=False, truncation=True, max_length=max_r)["input_ids"]
    B = tok(d["response_b"].tolist(), add_special_tokens=False, truncation=True, max_length=max_r)["input_ids"]
    return [H_P + p + H_A + a + H_B + b + H_Q for p, a, b in zip(P, A, B)]

In [ ]:
%%writefile /kaggle/working/feas/t_env.py
import sys; sys.path.insert(0, "/kaggle/working/feas")
from common import run

def main():
    import torch, vllm, transformers
    n = torch.cuda.device_count()
    return dict(vllm=vllm.__version__, torch=torch.__version__, transformers=transformers.__version__,
                gpus=[torch.cuda.get_device_name(i) for i in range(n)],
                compute_capability=torch.cuda.get_device_capability(0))
run("env", main)

In [ ]:
!cd /kaggle/working/feas && timeout 600 python t_env.py > logs/env.log 2>&1; tail -n 5 logs/env.log

In [ ]:
%%writefile /kaggle/working/feas/t_small_gen.py
import sys, time; sys.path.insert(0, "/kaggle/working/feas")
from common import run

def main():
    from vllm import LLM, SamplingParams
    t = time.time()
    llm = LLM(model="Qwen/Qwen2.5-0.5B-Instruct", dtype="half", max_model_len=2048,
              gpu_memory_utilization=0.85, enforce_eager=True)
    load_s = time.time() - t
    sp = SamplingParams(temperature=0, max_tokens=64)
    t = time.time()
    out = llm.generate(["Explain what a KV cache is in one sentence."] * 16, sp)
    dt = time.time() - t
    toks = sum(len(o.outputs[0].token_ids) for o in out)
    print("sample:", out[0].outputs[0].text[:200])
    return dict(load_s=round(load_s, 1), gen_tokens=toks, tok_per_s=round(toks / dt, 1))
run("small_gen", main)

In [ ]:
!cd /kaggle/working/feas && timeout 1200 python t_small_gen.py > logs/small_gen.log 2>&1; tail -n 6 logs/small_gen.log

In [ ]:
%%writefile /kaggle/working/feas/t_judge7b.py
import sys, time; sys.path.insert(0, "/kaggle/working/feas")
from common import run

TP = int(sys.argv[1]) if len(sys.argv) > 1 else 1
NAME = "judge7b" if TP == 1 else f"judge7b_tp{TP}"

def main():
    from vllm import LLM, SamplingParams
    t = time.time()
    llm = LLM(model="Qwen/Qwen2.5-7B-Instruct-AWQ", quantization="awq", dtype="half",
              max_model_len=4096, gpu_memory_utilization=0.90, tensor_parallel_size=TP,
              enforce_eager=True)
    load_s = time.time() - t
    q = ("A user asked: 'How do I reverse a list in Python?'\n\n"
         "Response A: Use my_list.reverse() to reverse in place, or my_list[::-1] for a reversed copy.\n\n"
         "Response B: You can't reverse lists in Python.\n\n"
         "Which response would a human prefer? Think in one short sentence, then answer with exactly one "
         "letter on the last line: A, B, or C (tie).")
    msgs = [[{"role": "user", "content": q}]] * 16
    t = time.time()
    out = llm.chat(msgs, SamplingParams(temperature=0, max_tokens=64))
    dt = time.time() - t
    toks = sum(len(o.outputs[0].token_ids) for o in out)
    print("sample:", out[0].outputs[0].text[:300])
    return dict(tp=TP, load_s=round(load_s, 1), gen_tokens=toks, tok_per_s=round(toks / dt, 1))
run(NAME, main)

In [ ]:
!cd /kaggle/working/feas && timeout 1800 python t_judge7b.py 1 > logs/judge7b.log 2>&1; tail -n 6 logs/judge7b.log

Optional: same 7B model split across both T4s (tensor parallelism). If this fails, Step 2 can still run the judge on one GPU.

In [ ]:
!cd /kaggle/working/feas && timeout 1800 python t_judge7b.py 2 > logs/judge7b_tp2.log 2>&1; tail -n 6 logs/judge7b_tp2.log

In [ ]:
%%writefile /kaggle/working/feas/t_classify.py
import sys, time, json; sys.path.insert(0, "/kaggle/working/feas")
from common import run, find, build_ids, OUT

def main():
    import os, numpy as np, pandas as pd
    from sklearn.metrics import log_loss
    from transformers import AutoTokenizer
    from vllm import LLM
    mdir = os.path.dirname(find("qwen-cls/config.json"))
    info = json.load(open(find("train_info.json")))
    val = pd.read_parquet(find("val.parquet")).iloc[:1000]
    tok = AutoTokenizer.from_pretrained(mdir)
    ids = build_ids(tok, val, info["max_p"], info["max_r"])

    llm, used, errors = None, None, []
    # vLLM API changed across versions: try the newer argument first, then the older one
    for kw in (dict(runner="pooling"), dict(task="classify")):
        try:
            llm = LLM(model=mdir, dtype="half", max_model_len=1024, gpu_memory_utilization=0.85,
                      enforce_eager=True, **kw)
            used = kw
            break
        except Exception as e:
            errors.append(f"{kw}: {repr(e)[:200]}")
    assert llm is not None, " | ".join(errors)

    prompts = [{"prompt_token_ids": x} for x in ids]
    t = time.time()
    outs = llm.classify(prompts)
    dt = time.time() - t
    probs = np.array([o.outputs.probs for o in outs], dtype=np.float32)
    np.save(f"{OUT}/vllm_probs_50.npy", probs[:50])
    ll = log_loss(val["label"].values, probs, labels=[0, 1, 2])
    return dict(loaded_with=used, rows=len(ids), samples_per_s=round(len(ids) / dt, 1),
                val_log_loss_1000=round(ll, 4), hf_ref_samples_per_s="13.0 (fp16 bs16 sorted, 1 GPU)")
run("classify", main)

In [ ]:
!cd /kaggle/working/feas && timeout 1800 python t_classify.py > logs/classify.log 2>&1; tail -n 6 logs/classify.log

In [ ]:
%%writefile /kaggle/working/feas/t_hf_match.py
import sys, json; sys.path.insert(0, "/kaggle/working/feas")
from common import run, find, build_ids, OUT

def main():
    import os, numpy as np, pandas as pd, torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    mdir = os.path.dirname(find("qwen-cls/config.json"))
    info = json.load(open(find("train_info.json")))
    val = pd.read_parquet(find("val.parquet")).iloc[:50]
    tok = AutoTokenizer.from_pretrained(mdir)
    ids = build_ids(tok, val, info["max_p"], info["max_r"])
    m = AutoModelForSequenceClassification.from_pretrained(mdir, dtype=torch.float16,
                                                           device_map={"": "cuda:0"}).eval()
    hf = []
    with torch.inference_mode():
        for x in ids:   # one by one: no padding, cleanest reference
            logits = m(input_ids=torch.tensor([x], device="cuda:0")).logits.float()
            hf.append(torch.softmax(logits, -1)[0].cpu().numpy())
    hf = np.array(hf)
    vl = np.load(f"{OUT}/vllm_probs_50.npy")
    diff = float(np.abs(hf - vl).max())
    same_argmax = float((hf.argmax(1) == vl.argmax(1)).mean())
    assert diff < 0.02, f"vLLM vs HF probabilities differ too much: max abs diff {diff:.4f}"
    return dict(rows=50, max_abs_prob_diff=round(diff, 5), same_argmax=same_argmax)
run("hf_match", main)

In [ ]:
!cd /kaggle/working/feas && timeout 900 python t_hf_match.py > logs/hf_match.log 2>&1; tail -n 6 logs/hf_match.log

## Summary

In [ ]:
import json, glob, pandas as pd
rows = [json.load(open(f)) for f in sorted(glob.glob("/kaggle/working/feas/*.json"))]
summary = pd.DataFrame(rows)
order = ["env", "small_gen", "judge7b", "judge7b_tp2", "classify", "hf_match"]
summary["o"] = summary["test"].map({k: i for i, k in enumerate(order)})
summary = summary.sort_values("o").drop(columns="o").reset_index(drop=True)
pd.set_option("display.max_colwidth", 120)
display(summary)
summary.to_csv("/kaggle/working/feasibility_summary.csv", index=False)